### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [31]:
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [32]:
from pathlib import Path
folder= Path().resolve().parent /'input'


### 1. Load multiple dataset for different buildings.

In [33]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / 'load_profile_buildingID_*')

In [34]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [35]:
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
meta_data=(
    pl.scan_parquet(folder / 'MetaData.parquet')
)            
meta_data.select(pl.col('in.ashrae_iecc_climate_zone_2004_sub_cz_split')).unique().collect().to_series().to_list()

['4A',
 '3A',
 '2B',
 '8AK',
 '2A - FL, GA, AL, MS',
 '7B',
 '3B',
 '7A',
 '1A - HI',
 '2A - TX, LA',
 '5A',
 '6B',
 '1A - FL',
 '4B',
 '3C',
 '4C',
 '5B',
 '7AK',
 '6A']

In [36]:
MetaData=meta_data.filter(
            pl.col('bldg_id').is_in(bldgs)).cast({"bldg_id": pl.UInt32}).select(
            pl.col('bldg_id', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'))

# MetaData.row(0, named=True) 

In [37]:
%%time
# merge the climate zone changes into the original data
df=data.join(MetaData, on='bldg_id')
df.collect()

CPU times: user 433 ms, sys: 147 ms, total: 580 ms
Wall time: 167 ms


bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,out.electricity.heating.energy_consumption..kwh,out.electricity.heating_fans_pumps.energy_consumption..kwh,out.electricity.heating_hp_bkup.energy_consumption..kwh,out.electricity.heating_hp_bkup_fa.energy_consumption..kwh,out.electricity.hot_water.energy_consumption..kwh,out.electricity.hot_water_solar_th.energy_consumption..kwh,out.electricity.lighting_exterior.energy_consumption..kwh,out.electricity.lighting_garage.energy_consumption..kwh,out.electricity.lighting_interior.energy_consumption..kwh,out.electricity.mech_vent.energy_consumption..kwh,out.electricity.net.energy_consumption..kwh,out.electricity.permanent_spa_heat.energy_consumption..kwh,out.electricity.permanent_spa_pump.energy_consumption..kwh,out.electricity.plug_loads.energy_consumption..kwh,out.electricity.pool_heater.energy_consumption..kwh,out.electricity.pool_pump.energy_consumption..kwh,out.electricity.pv.energy_consumption..kwh,out.electricity.range_oven.energy_consumption..kwh,out.electricity.refrigerator.energy_consumption..kwh,out.electricity.television.energy_consumption..kwh,out.electricity.total.energy_consumption..kwh,out.electricity.well_pump.energy_consumption..kwh,out.fuel_oil.heating.energy_consumption..kwh,out.fuel_oil.hot_water.energy_consumption..kwh,out.fuel_oil.total.energy_consumption..kwh,out.natural_gas.clothes_dryer.energy_consumption..kwh,…,out.indoor_operative_temperature.conditioned_space..c,out.indoor_radiant_temperature.conditioned_space..c,out.indoor_relative_humidity.conditioned_space..percentage,out.indoor_temperature.conditioned_space..c,out.outdoor_air_drybulb_temp..c,out.outdoor_air_relative_humidity..percentage,out.outdoor_air_wetbulb_temp..c,out.outdoor_humidity_ratio..kgwater_per_kgdryair,out.people.sensible_heating_rate..watt,out.people.total_heating_rate..watt,out.weather.diffuse_solar_radiation..watt_per_m2,out.weather.direct_normal_solar_radiation..watt_per_m2,out.weather.wind_speed..meter_per_second,out.schedules.ceiling_fan,out.schedules.clothes_dryer,out.schedules.clothes_washer,out.schedules.cooking_range,out.schedules.cooling_setpoint..c,out.schedules.dishwasher,out.schedules.electric_vehicle_charging,out.schedules.electric_vehicle_discharging,out.schedules.heating_setpoint..c,out.schedules.hot_water_clothes_washer,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_garage,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.peak_period,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.pre_peak_period,out.schedules.vacancy,in.ashrae_iecc_climate_zone_2004_sub_cz_split
u32,datetime[ms],f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str
10,2018-01-01 00:15:00,1228.0,0.00514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.05603,0.00548,0.0,0.0,0.0,0.0,0.00088,0.0,0.00497,0.0,0.1662,0.0,0.0,0.05554,0.0,0.0,0.0,0.0,0.01194,0.01312,0.1662,0.01309,0.0,0.0,0.0,0.0,…,24.52861,24.335056,51.607399,24.722221,17.25,85.660004,15.726111,0.010475,132.006317,234.456863,0.0,0.0,4.19993,0.566,0.0,0.0,0.0,24.7222,null,null,null,24.7222,0.0,null,0.0,null,0.0993,0.0,0.0,0.5,0.0,0.601,0.0699,0.0,0.0,0.0,"""1A - FL"""
10,2018-01-01 00:30:00,1228.0,0.00514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.04812,0.00467,0.0,0.0,0.0,0.0,0.0008

There are only time two climate zones for the data

----------------------------------------------

## Explore the data to get insights

In [38]:
import polars.selectors as pls
df=df.select(pls.matches(r'bldg_id|timestamp|in.ashrae_iecc_climate_zone_*|out.electricity.*?.energy_consumption..kwh'))

## Preprocess the data

In [39]:
# defining the independant and dependant variables with encoding categorical variables and converting the datetime to a timestamp
raw_x= df.select(pl.col("timestamp").dt.timestamp(), pl.col('out.electricity.total.energy_consumption..kwh').alias('Total Usage'),
             pl.col('in.ashrae_iecc_climate_zone_2004_sub_cz_split').cast(pl.Categorical).cast(pl.UInt32).alias('Climate Zone')).collect()
y=df.select(pl.col("out.electricity.coobling.energy_consumption..kwh").alias('Cooling Consumption'),
           pl.col("out.electricity.heating.energy_consumption..kwh").alias('Heating Consumption'),
           pl.col("out.electricity.freezer.energy_consumption..kwh").alias('Freezing Consumption'),
           pl.col("out.electricity.refrigerator.energy_consumption..kwh").alias('Refrigerator Consumption'),
           pl.col("out.electricity.dishwasher.energy_consumption..kwh").alias('Dishwasher Consumption')).collect()

ColumnNotFoundError: out.electricity.coobling.energy_consumption..kwh

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
SELECT [col("bldg_id"), col("timestamp"), col("out.electricity.ceiling_fan.energy_consumption..kwh"), col("out.electricity.clothes_dryer.energy_consumption..kwh"), col("out.electricity.clothes_washer.energy_consumption..kwh"), col("out.electricity.cooling.energy_consumption..kwh"), col("out.electricity.cooling_fans_pumps.energy_consumption..kwh"), col("out.electricity.dishwasher.energy_consumption..kwh"), col("out.electricity.ev_charging.energy_consumption..kwh"), col("out.electricity.freezer.energy_consumption..kwh"), col("out.electricity.heating.energy_consumption..kwh"), col("out.electricity.heating_fans_pumps.energy_consumption..kwh"), col("out.electricity.heating_hp_bkup.energy_consumption..kwh"), col("out.electricity.heating_hp_bkup_fa.energy_consumption..kwh"), col("out.electricity.hot_water.energy_consumption..kwh"), col("out.electricity.h

In [40]:
raw_x.select(pl.col('Total Usage').min())

Total Usage
f32
0.04306


## test standardizing the data with the standard scaler

In [66]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x=scaler.fit_transform(raw_x)

In [73]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    return r2_score(y_test, predicted)
        # r2_score(y_test, predicted)
        # mean_absolute_error(y_test, predicted), 
        # mean_squared_error(y_test, predicted), 
        # root_mean_squared_error(y_test, predicted)

## test standardizing the data with the Min Max Scaler

In [72]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
x=scaler.fit_transform(raw_x)

### Prepare cross-validation model


In [73]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    return r2_score(y_test, predicted)
        # r2_score(y_test, predicted)
        # mean_absolute_error(y_test, predicted), 
        # mean_squared_error(y_test, predicted), 
        # root_mean_squared_error(y_test, predicted)

In [74]:
%%time
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg
skf= StratifiedKFold(n_splits=6)
kf=KFold(n_splits=10)
arr=[]
lf=pd.DataFrame()
for i, (train,test) in enumerate(kf.split(x,y)):
    x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
    LR=model(LinearRegression(), x_train, x_test, y_train, y_test)
    RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test)
    XG=model(xg.XGBRegressor(), x_train, x_test, y_train, y_test)
    frame=pd.DataFrame({
    "Split Number": [i],
    "Linear Regression": [LR],
    "Random Forest": [RF],
    "XG Boost": [XG]
    })
    arr.append(frame)
lf=pd.concat(arr)

CPU times: user 4min 9s, sys: 734 ms, total: 4min 10s
Wall time: 16.2 s


## Calculating measures

In [76]:
lf.mean()

Split Number         4.500000
Linear Regression   -0.293474
Random Forest       -0.268126
XG Boost            -0.851046
dtype: float64

In [16]:
# prepare measurement
Test_data=x.head(1)
Actual_data=y.row(1)

In [17]:
Test_data

timestamp,Total Usage,Climate Zone
i64,f32,u32
1514765700000000,0.1662,0


In [18]:
Actual_data

(0.0, 0.04811999946832657, 0.0, 0.011939999647438526, 0.0)

In [19]:
# Select the best model 
import xgboost as xg
from sklearn.model_selection import train_test_split
x,x_test, y, y_test= train_test_split(x,y, test_size=.3, random_state=42)
xg_model=xg.XGBRegressor()
xg_model.fit(x,y)
estimated_values=xg_model.predict(Test_data)

In [24]:
estimated_values.flatten()

array([1.2082888e-02, 4.1609019e-02, 1.0798807e-17, 1.1186296e-02,
       1.5908480e-04], dtype=float32)

## Observation

Apparently using r square as a metric to represent to the data is a misrepresentation to the model's performance.
R square formula is - >1- ( total of squared residulas /total of squared standard deviation)
Knowing MAE is the total of residuals -> total(y - yPredicted)
giving r square = 0.02 therefore
giving standard deviation of .001 as given above
then r square= .0004 / .00001= 40 -> 1-40 =-39 which is wrong

Therefore, MAE score is the perfect representation of the data.
another testing metric should be provided.

Input data = timestamp Total      Usage     Climate Zone.
             1545212700000000     1.22827       1

Actual result=(0.1280200034379959, 0.0, 0.0, 0.01245999988168478, 0.0).


Estimated result= [2.9516332e-02, 4.0768548e-03, 1.0928856e-17, 1.0773698e-02,
       2.7552096e-03].